In [1]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/naver/disk2/HoangDPB/rPPG-Toolbox"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.FactorizePhys.FactorizePhys import FactorizePhys

In [ ]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "raw_data/5_demo")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/ppg/5_demo/UBFC-rPPG_FactorizePhys")
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/ppg/5_demo/UBFC-rPPG_FactorizePhys")

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 25       # PPG sensor sampling rate (Hz)

# ----- preprocessing params -----
CHUNK_LENGTH = 160     # frames per clip
IMG_H, IMG_W = 72, 72  # face crop resolution
LABEL_TYPE   = "Standardized"  # z-score label (no cumsum in post-processing)

# ----- PPG denoising -----
# 'imat'    : IMAT (Mashhadi 2015) - SVD + LMS + iterative sparse reconstruction (try this)
# 'wiener'  : Windowed Wiener filter (Temko 2017) - FFT-domain noise suppression
# 'kalman'  : Windowed SVD + Kalman (Seyedtabaii 2008) - adaptive smoothing
# 'bandpass': bandpass filter only (0.75-4 Hz)
# 'none'    : raw green channel (original behavior)
DENOISE_METHOD = "imat"

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Denoise method:", DENOISE_METHOD)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [3]:
# Model
MODEL_LABEL    = "UBFC-rPPG_FactorizePhys"
MODEL_REL_PATH = "final_model_release/UBFC-rPPG_FactorizePhys_FSAM_Res.pth"
MODEL_PATH     = os.path.join(REPO_ROOT, MODEL_REL_PATH)

print(f"Model: {MODEL_LABEL}")
print(f"Path:  {MODEL_PATH}")

Model: UBFC-rPPG_FactorizePhys
Path:  /home/naver/disk2/HoangDPB/rPPG-Toolbox/final_model_release/UBFC-rPPG_FactorizePhys_FSAM_Res.pth


In [4]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [5]:
# Read and time-align PPG signal

def read_ppg_synced(session_path, num_frames, fps=30):
    """Read PPG green channel from ppg.csv and resample to video frame times.

    The PPG sensor timestamps and video start timestamp are both in ms
    Unix epoch stored in metadata.json (sync_markers.video_start).
    Resampling is done by linear interpolation onto frame-spaced timestamps.

    Returns:
        ppg (np.ndarray): shape (num_frames,), dtype float32, raw green values.
    """
    meta_path = os.path.join(session_path, "metadata.json")
    with open(meta_path, "r") as f:
        meta = json.load(f)
    video_start_ms = meta["sync_markers"].get("video_start")
    if video_start_ms is None:
        video_start_ms = meta.get("start_timestamp")
    if video_start_ms is None:
        raise ValueError(f"No video_start or start_timestamp found in {meta_path}")
    video_start_ms = float(video_start_ms)

    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_ts    = ppg_df["timestamp"].values.astype(np.float64)
    ppg_green = ppg_df["green"].values.astype(np.float64)

    # convert ppg timestamps to seconds relative to video start
    ppg_t = (ppg_ts - video_start_ms) / 1000.0

    # frame timestamps in seconds
    frame_t = np.arange(num_frames, dtype=np.float64) / fps

    # clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])

    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_green)
    return ppg_resampled.astype(np.float32)

In [ ]:
# ===== PPG Denoising using Accelerometer Data (Windowed Processing) =====
# Adapted from GalaxyPPG-Supplementary-Code/Step03_Denosing/
# All methods process in 8-second windows (matching the original design).
#
# References:
#   - Mashhadi et al. (2015) "Heart rate tracking using wrist-type PPG signals" [IMAT]
#   - Temko (2017) "Accurate heart rate monitoring during physical exercises" [Wiener]
#   - Seyedtabaii et al. (2008) "Kalman filter based adaptive reduction of motion artifact"
#   - Reddy & Kumar (2007) "Motion artifact reduction in PPG using SVD"


def read_acc_aligned(session_path, ppg_timestamps_ms):
    """Read acc.csv and interpolate x/y/z to PPG timestamps."""
    acc_df = pd.read_csv(os.path.join(session_path, "acc.csv"))
    acc_ts = acc_df["timestamp"].values.astype(np.float64)
    return np.column_stack([
        np.interp(ppg_timestamps_ms, acc_ts, acc_df[c].values.astype(np.float64))
        for c in ("x", "y", "z")
    ])


# ---------- helpers shared by all methods ----------

def _normalize(x):
    return (x - np.mean(x)) / (np.std(x) + 1e-10)

def _minmax(x):
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo + 1e-10)

def _bandpass(sig, fs, low=0.75, high=4.0, order=2):
    nyq = 0.5 * fs
    b, a = signal.butter(order, [low / nyq, high / nyq], btype="band")
    return signal.filtfilt(b, a, np.float64(sig))


# ---------- IMAT (Mashhadi 2015) — reimplemented ----------
# Uses np.linalg.eigh (safe for symmetric matrices) instead of np.linalg.eig
# to avoid NumPy 1.22 complex array memory corruption.

class IMATDenoiser:
    """IMAT motion artifact removal for wearable PPG signals."""

    def __init__(self, fs=25, target_length=200):
        self.fs = fs
        self.target_length = target_length
        self.window_count = 0
        self.Nprv = 0.0
        self.previous_ppg = None
        self.previous_acc_x = None
        self.previous_acc_y = None
        self.previous_acc_z = None

    def process_window(self, ppg, acc_x, acc_y, acc_z):
        """Process one 8-second window. Returns (denoised_signal, bpm)."""
        tl = self.target_length
        def pad_or_trim(sig):
            if len(sig) < tl: return np.pad(sig, (0, tl - len(sig)), mode='edge')
            return sig[:tl]

        self.window_count += 1
        ppg = np.array(pad_or_trim(ppg), dtype=np.float64)
        acc_x = np.array(pad_or_trim(acc_x), dtype=np.float64)
        acc_y = np.array(pad_or_trim(acc_y), dtype=np.float64)
        acc_z = np.array(pad_or_trim(acc_z), dtype=np.float64)

        if self.window_count == 1:
            self.Nprv = self._init_hr(ppg) * 4
            x3 = ppg.copy()
            self.previous_ppg = ppg.copy()
            self.previous_acc_x = acc_x.copy()
            self.previous_acc_y = acc_y.copy()
            self.previous_acc_z = acc_z.copy()
        else:
            x2 = self._ma_cancellation(ppg, acc_x, acc_y, acc_z, 30, 10, self.Nprv)
            ext_ppg = np.concatenate([self.previous_ppg, ppg])
            ext_ax = np.concatenate([self.previous_acc_x, acc_x])
            ext_ay = np.concatenate([self.previous_acc_y, acc_y])
            ext_az = np.concatenate([self.previous_acc_z, acc_z])
            z2 = self._ma_cancellation(ext_ppg, ext_ax, ext_ay, ext_az, 30, 10, self.Nprv)
            ml = min(len(x2), len(z2))
            x3 = (x2[:ml] + z2[:ml]) / 2
            self.previous_ppg = ppg.copy()
            self.previous_acc_x = acc_x.copy()
            self.previous_acc_y = acc_y.copy()
            self.previous_acc_z = acc_z.copy()

        bpm = self._estimate_hr(x3)
        self.Nprv = bpm * 4
        return x3, bpm

    def _init_hr(self, ppg):
        f = np.abs(np.fft.fft(ppg))
        freqs = np.fft.fftfreq(len(ppg), 1.0 / self.fs)
        mask = (freqs >= 1.2) & (freqs <= 1.5)
        if not mask.any(): return 75.0
        return float(freqs[mask][np.argmax(f[mask])]) * 60

    def _estimate_hr(self, sig):
        sig = np.ascontiguousarray(sig, dtype=np.float64)
        N = max(2048, len(sig))
        f = np.abs(np.fft.fft(sig, N))
        freqs = np.fft.fftfreq(N, 1.0 / self.fs)
        mask = (freqs >= 0.75) & (freqs <= 3.5)
        if not mask.any(): return 75.0
        return float(freqs[mask][np.argmax(f[mask])]) * 60

    def _ma_cancellation(self, ppg, acc_x, acc_y, acc_z, k, deg, Nprv):
        tool = len(ppg)
        y1 = self._ref_ma_svd(acc_x, k, tool)
        y2 = self._ref_ma_svd(acc_y, k, tool)
        y3 = self._ref_ma_svd(acc_z, k, tool)
        s1 = np.ascontiguousarray(y1.reshape(k, tool).T)
        s2 = np.ascontiguousarray(y2.reshape(k, tool).T)
        s3 = np.ascontiguousarray(y3.reshape(k, tool).T)
        x2 = ppg - np.mean(ppg)
        en = np.sqrt(2 * np.mean(x2 ** 2)) * 2
        x2 = np.clip(x2, -en, en)
        return self._adaptive_lms(s1, s2, s3, x2, k, deg, Nprv)

    def _ref_ma_svd(self, sig, L, tool):
        """SVD reference generation using eigh (safe for symmetric matrices)."""
        N = tool
        if L > N // 2: L = N - L
        K = N - L + 1
        X = np.zeros((L, K), dtype=np.float64)
        for i in range(K): X[:, i] = sig[i:i + L]
        S = X @ X.T
        eigenvals, U = np.linalg.eigh(S)
        idx = np.argsort(-eigenvals); U = U[:, idx]
        V = X.T @ U
        Lp, Kp = min(L, K), max(L, K)
        p = []
        for j in range(min(L, K)):
            rca = np.outer(U[:, j], V[:, j])
            y = np.zeros(N, dtype=np.float64)
            for kk in range(Lp - 1):
                s = sum(rca[m, kk - m] for m in range(kk + 1) if m < rca.shape[0] and kk - m < rca.shape[1])
                y[kk] = s / (kk + 1)
            for kk in range(Lp - 1, Kp):
                s = sum(rca[m, kk - m] for m in range(Lp) if m < rca.shape[0] and kk - m < rca.shape[1])
                y[kk] = s / Lp
            for kk in range(Kp, N):
                s = sum(rca[m, kk - m] for m in range(kk - Kp + 2, min(N - Kp + 1, rca.shape[0])) if m < rca.shape[0] and kk - m < rca.shape[1])
                y[kk] = s / max(1, N - kk)
            p.append(y)
        return np.concatenate(p)

    def _adaptive_lms(self, s1, s2, s3, x2, k, deg, Nprv):
        ms = 0.005; fft_length = 6000
        def get_freq_peak(col):
            f = np.abs(np.fft.fft(np.ascontiguousarray(col, dtype=np.float64), fft_length))
            return int(np.argmax(f[:800]))
        k = min(20, k)
        frqs1 = np.array([get_freq_peak(s1[:, i]) for i in range(min(k, s1.shape[1]))])
        frqs2 = np.array([get_freq_peak(s2[:, i]) for i in range(min(k, s2.shape[1]))])
        frqs3 = np.array([get_freq_peak(s3[:, i]) for i in range(min(k, s3.shape[1]))])
        def lms_filter(x_ref, d, flen, step):
            N = len(x_ref); w = np.zeros(flen, dtype=np.float64); y = np.zeros(N, dtype=np.float64)
            xr = np.ascontiguousarray(x_ref, dtype=np.float64); dd = np.ascontiguousarray(d, dtype=np.float64)
            for n in range(flen, N):
                xv = xr[n - flen:n][::-1].copy(); y[n] = np.dot(w, xv)
                w = w + 2 * step * (dd[n] - y[n]) * xv
            return y
        for refs, frqs in [(s1, frqs1), (s2, frqs2), (s3, frqs3)]:
            for i in range(min(k, refs.shape[1])):
                ref_sig = np.ascontiguousarray(refs[:, i], dtype=np.float64)
                x2 = x2 - lms_filter(ref_sig, x2, deg, ms)
        return x2


def denoise_imat(ppg, acc, fs=25):
    """Denoise full PPG signal using IMAT (SVD + LMS adaptive filter).
    Non-overlapping 200-sample windows, matching the original GalaxyPPG pipeline.
    """
    win_len = 200; N = len(ppg)
    ppg_f = _bandpass(ppg, fs, low=0.5, high=4.0, order=4)
    acc_f = np.column_stack([_bandpass(acc[:, i], fs, low=0.5, high=4.0, order=4) for i in range(3)])
    ppg_norm = _minmax(ppg_f)
    acc_norm = np.column_stack([_minmax(acc_f[:, i]) for i in range(3)])
    denoiser = IMATDenoiser(fs=fs, target_length=win_len)
    parts = []
    start = 0
    while start + win_len <= N:
        d, _ = denoiser.process_window(ppg_norm[start:start+win_len], acc_norm[start:start+win_len,0], acc_norm[start:start+win_len,1], acc_norm[start:start+win_len,2])
        parts.append(np.asarray(d, dtype=np.float64)); start += win_len
    if start < N:
        rem = N - start
        wp = np.pad(ppg_norm[start:], (0, win_len-rem), mode='edge')
        wa = [np.pad(acc_norm[start:,i], (0, win_len-rem), mode='edge') for i in range(3)]
        d, _ = denoiser.process_window(wp, wa[0], wa[1], wa[2])
        parts.append(np.asarray(d, dtype=np.float64)[:rem])
    return np.concatenate(parts)


# ---------- Wiener filter (Temko 2017) ----------

def _wiener_window(ppg_win, acc_win, state, fs=25):
    """Single-window Wiener filter in FFT domain 1-3 Hz."""
    fft_res = 1024; wf_length = 15; low_hz, high_hz = 1.0, 3.0
    ppg_f = _bandpass(ppg_win, fs, low=low_hz, high=high_hz, order=4)
    acc_f = np.column_stack([_bandpass(acc_win[:, i], fs, low=low_hz, high=high_hz, order=4) for i in range(3)])
    ppg_norm = _minmax(ppg_f); acc_norm = np.column_stack([_minmax(acc_f[:, i]) for i in range(3)])
    ppg_fft = np.fft.fft(ppg_norm, fft_res)
    acc_ffts = [np.fft.fft(acc_norm[:, i], fft_res) for i in range(3)]
    freq = np.linspace(0, fs, fft_res)
    lo_i = np.argmin(np.abs(freq - low_hz)); hi_i = np.argmin(np.abs(freq - high_hz))
    fr = freq[lo_i:hi_i]; pf = ppg_fft[lo_i:hi_i]; afs = [a[lo_i:hi_i] for a in acc_ffts]
    fr_ppg = fr.copy()
    if state["prev_ppg_fft"] is not None:
        for ii in range(len(fr_ppg)):
            cur_ph = np.angle(pf[ii]); prev_ph = np.angle(state["prev_ppg_fft"][ii])
            vocoder = np.array([((cur_ph - prev_ph) + 2*np.pi*n) / (2*np.pi*2) for n in range(20)])
            fr_ppg[ii] = vocoder[np.argmin(np.abs(vocoder - fr[ii]))]
    fr_ppg = np.convolve(fr_ppg, np.ones(3)/3, mode="same")
    def _hist_avg(lst, wl):
        if len(lst) < wl: return np.mean(lst, axis=0)
        return np.mean(lst[-wl:], axis=0)
    acc_avg = sum(np.abs(a)/(np.max(np.abs(a))+1e-10) for a in afs) / 3
    w1 = np.abs(pf)/(np.max(np.abs(pf))+1e-10)
    w1_avg = _hist_avg(state["w1_hist"]+[w1], wf_length); w1_norm = w1_avg/(np.max(w1_avg)+1e-10)
    wf1 = 1 - acc_avg/(w1_norm+1e-10); wf1[wf1<0] = -1; clean1 = np.abs(pf)*wf1
    w2 = np.abs(pf)/(np.max(np.abs(pf))+1e-10)
    w2_avg = _hist_avg(state["w2_hist"]+[w2], wf_length); w2_norm = w2_avg/(np.max(w2_avg)+1e-10)
    wf2 = w2_norm/(acc_avg+w2_norm+1e-10); clean2 = np.abs(pf)*wf2
    clean1 /= (np.std(clean1)+1e-10); clean2 /= (np.std(clean2)+1e-10); combined = clean1+clean2
    hist_int = 25
    if len(state["prev_bpm"])>15: hist_int = max(np.abs(np.diff(state["prev_bpm"])))+5
    freq_res = fr[1]-fr[0] if len(fr)>1 else 1; rw = int(hist_int/(freq_res*60+1e-10))
    if state["range_idx"] is None: idx = int(np.argmax(combined))
    else:
        valid = state["range_idx"][state["range_idx"]<len(combined)]
        idx = int(valid[np.argmax(combined[valid])]) if len(valid) else int(np.argmax(combined))
    bpm_est = fr_ppg[idx]*60; range_idx = np.arange(max(0,idx-rw), min(len(fr_ppg),idx+rw))
    if len(state["prev_bpm"])>5 and abs(bpm_est-state["prev_bpm"][-1])>5:
        recent = np.array(state["prev_bpm"][-5:])
        predicted = np.polyval(np.polyfit(range(len(recent)), recent, 1), len(recent))
        bpm_est = 0.8*bpm_est + 0.2*predicted
    if len(state["prev_bpm"])>6:
        bpm_est += np.sum(np.sign(np.array(state["prev_bpm"][-6:])-np.array(state["prev_bpm"][-7:-1]))*0.1)
    full_fft = np.zeros(fft_res, dtype=complex); full_fft[lo_i:hi_i] = combined*np.exp(1j*np.angle(pf))
    denoised = np.real(np.fft.ifft(full_fft))[:len(ppg_win)]
    new_state = {"prev_ppg_fft": pf, "w1_hist": (state["w1_hist"]+[w1])[-30:],
                 "w2_hist": (state["w2_hist"]+[w2])[-30:], "prev_bpm": (state["prev_bpm"]+[bpm_est])[-30:],
                 "range_idx": range_idx}
    return denoised, bpm_est, new_state

def denoise_wiener(ppg, acc, fs=25, win_sec=8.0, step_sec=2.0):
    win_len = int(win_sec*fs); step_len = int(step_sec*fs); N = len(ppg)
    denoised = np.zeros(N); weights = np.zeros(N); hann = np.hanning(win_len)
    state = {"prev_ppg_fft": None, "w1_hist": [], "w2_hist": [], "prev_bpm": [], "range_idx": None}
    start = 0
    while start+win_len <= N:
        clean, _, state = _wiener_window(ppg[start:start+win_len], acc[start:start+win_len], state, fs)
        denoised[start:start+win_len] += clean*hann; weights[start:start+win_len] += hann; start += step_len
    mask = weights > 1e-10; denoised[mask] /= weights[mask]
    if not mask.all(): denoised[~mask] = _bandpass(ppg, fs)[~mask]
    return denoised


# ---------- SVD + Kalman (windowed) ----------

def _svd_kalman_window(ppg_win, acc_win, fs=25):
    svd_threshold = 0.4; Q = 0.1
    ppg_n = _normalize(ppg_win); acc_n = np.apply_along_axis(_normalize, 0, acc_win)
    ppg_f = _bandpass(ppg_n, fs, low=0.5, high=4.0)
    N = len(ppg_f); L = N//2; K = N-L+1
    def hankel(sig):
        H = np.zeros((L, K))
        for i in range(K): H[:, i] = sig[i:i+L]
        return H
    def auto_thresh(evals):
        n = min(10, len(evals))
        if n < 2: return max(4, n)
        d = -np.diff(evals[:n]); idx = np.where(d > np.mean(d))[0]
        if len(idx) and d[idx[-1]] > 2*(evals[0]-evals[1]): return max(4, idx[-1]+1)
        return n
    H_ppg = hankel(ppg_f); U, S, Vt = np.linalg.svd(H_ppg, full_matrices=False); ppg_aut = auto_thresh(S)
    max_corr = 0.0
    for ax in range(3):
        H_a = hankel(acc_n[:, ax]); Ua, Sa, _ = np.linalg.svd(H_a, full_matrices=False); aa = auto_thresh(Sa)
        c = (U[:, :ppg_aut].T @ Ua[:, :aa])**2; max_corr = max(max_corr, float(np.max(np.sum(c, axis=1))))
    Sf = S.copy()
    if max_corr > svd_threshold: Sf *= (1-max_corr)
    Hf = (U*Sf[np.newaxis,:]) @ Vt; out = np.zeros(N); cnt = np.zeros(N)
    for i in range(K): out[i:i+L] += Hf[:, i]; cnt[i:i+L] += 1
    svd_out = out/np.maximum(cnt, 1)
    acc_mag = np.sqrt(np.sum(acc_n**2, axis=1)); acc_th = np.mean(acc_mag)+2*np.std(acc_mag)
    dt = 1.0/fs; F = np.array([[1,dt],[0,1]]); Ho = np.array([[1.0,0.0]])
    x = np.array([svd_out[0], 0.0]); P = np.eye(2)*1000.0; filtered = np.zeros(N)
    for i in range(N):
        R = 10.0 if acc_mag[i] > acc_th else 1.0
        xp = F@x; Pp = F@P@F.T+np.eye(2)*Q; inn = svd_out[i]-Ho@xp; Si = Ho@Pp@Ho.T+R
        Kg = Pp@Ho.T/Si; x = xp+Kg.reshape(-1)*float(inn); P = (np.eye(2)-Kg.reshape(-1,1)@Ho)@Pp
        filtered[i] = x[0]
    return filtered

def denoise_kalman(ppg, acc, fs=25, win_sec=8.0, step_sec=2.0):
    win_len = int(win_sec*fs); step_len = int(step_sec*fs); N = len(ppg)
    denoised = np.zeros(N); weights = np.zeros(N); hann = np.hanning(win_len)
    start = 0
    while start+win_len <= N:
        clean = _svd_kalman_window(ppg[start:start+win_len], acc[start:start+win_len], fs)
        denoised[start:start+win_len] += clean*hann; weights[start:start+win_len] += hann; start += step_len
    mask = weights > 1e-10; denoised[mask] /= weights[mask]
    if not mask.all(): denoised[~mask] = _bandpass(ppg, fs)[~mask]
    return denoised


# ---------- Main entry point ----------

def read_ppg_denoised(session_path, num_frames, fps=30, ppg_fs=25, method="wiener"):
    """Read PPG green channel, denoise with accelerometer, resample to video frames."""
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_ts    = ppg_df["timestamp"].values.astype(np.float64)
    ppg_green = ppg_df["green"].values.astype(np.float64)
    if method in ("wiener", "kalman", "imat"):
        acc_data = read_acc_aligned(session_path, ppg_ts)
        if method == "wiener":   ppg_clean = denoise_wiener(ppg_green, acc_data, fs=ppg_fs)
        elif method == "kalman": ppg_clean = denoise_kalman(ppg_green, acc_data, fs=ppg_fs)
        else:                    ppg_clean = denoise_imat(ppg_green, acc_data, fs=ppg_fs)
    elif method == "bandpass":
        ppg_clean = _bandpass(ppg_green, ppg_fs, low=0.75, high=4.0)
    else:
        ppg_clean = ppg_green
    meta_path = os.path.join(session_path, "metadata.json")
    with open(meta_path) as f: meta = json.load(f)
    video_start_ms = meta["sync_markers"].get("video_start") or meta.get("start_timestamp")
    video_start_ms = float(video_start_ms)
    ppg_t = (ppg_ts - video_start_ms) / 1000.0
    frame_t = np.arange(num_frames, dtype=np.float64) / fps
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    return np.interp(frame_t_clipped, ppg_t, ppg_clean).astype(np.float32)


print(f"PPG denoising ready: method={DENOISE_METHOD}")
print(f"  imat   : 8s non-overlapping windows, SVD+LMS+iterative sparse reconstruction")
print(f"  wiener : 8s windows, 2s step, Wiener filter 1-3 Hz")
print(f"  kalman : 8s windows, 2s step, SVD+Kalman 0.5-4 Hz")

In [7]:
# Normalization functions
# DATA_TYPE is Raw: no normalization on input frames (just face crop + resize)
# LABEL_TYPE is Standardized: z-score normalization on labels

def standardized_label(label):
    """Standardized label: global z-score."""
    label = label.astype(np.float64)
    m = np.mean(label)
    s = np.std(label)
    if s > 0:
        label = (label - m) / s
    else:
        label = np.zeros_like(label)
    return label.astype(np.float32)

In [8]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [9]:
# Discover subjects

all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_dirs = glob.glob(os.path.join(RAW_DATA_PATH, subj_id, "*"))
    if not session_dirs:
        print(f"No session folder for {subj_id}, skipping.")
        continue
    session_path = session_dirs[0]

    session_name  = os.path.basename(session_path)
    video_pattern = os.path.join(RAW_DATA_PATH, "videos",
                                  f"{subj_id}_{session_name}.mp4")
    video_files = glob.glob(video_pattern)
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
    })
    print(f"  {subj_id}  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

Found 5 subject folders

  S_000  video=S_000_baseline_1768465796513.mp4
  S_001  video=S_001_baseline_1768467220962.mp4
  S_002  video=S_002_baseline_1768467567134.mp4
  S_003  video=S_003_baseline_1768468115471.mp4
  S_004  video=S_004_baseline_1768468421043.mp4

Total subjects: 5


In [10]:
# Data preprocessing
# DATA_TYPE = Raw: face crop + resize only, NO normalization on input frames
# LABEL_TYPE = Standardized: z-score on labels
# PPG denoising: controlled by DENOISE_METHOD ('kalman', 'bandpass', or 'none')
# Save as .npy with shape (CHUNK_LENGTH, H, W, 3)

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}")
print(f"Denoise method: {DENOISE_METHOD}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    # Use denoised PPG as ground truth label
    ppg_signal = read_ppg_denoised(
        session_path, T, fps=VIDEO_FPS, ppg_fs=PPG_FS, method=DENOISE_METHOD
    )
    print(f"  PPG ({DENOISE_METHOD}): min={ppg_signal.min():.4f}, max={ppg_signal.max():.4f}")

    # Face crop + resize (Raw data type: no normalization on frames)
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)  # (T, 72, 72, 3) float32

    # Standardized label (z-score)
    label = standardized_label(ppg_signal)  # (T,)

    # Chunk into clips of CHUNK_LENGTH
    clip_num = T // CHUNK_LENGTH
    frame_clips = np.array([frames_cropped[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH] for i in range(clip_num)])

    # Save per-subject subfolder
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, frame_clips[chunk_idx])  # (CHUNK_LENGTH, H, W, 3)
        np.save(label_path, label_clips[chunk_idx])   # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

Cleared and recreated: /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/ppg/5_demo/UBFC-rPPG_FactorizePhys
Denoise method: wiener

=== Processing S000 ===
  Video: 2704 frames @ 30 fps
  PPG (wiener): min=-404.3138, max=0.0673
  16 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/ppg/5_demo/UBFC-rPPG_FactorizePhys/S000

=== Processing S001 ===
  Video: 2748 frames @ 30 fps
  PPG (wiener): min=-0.0634, max=360.0654
  17 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/ppg/5_demo/UBFC-rPPG_FactorizePhys/S001

=== Processing S002 ===
  Video: 2739 frames @ 30 fps
  PPG (wiener): min=-1673.1438, max=0.0996
  17 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/ppg/5_demo/UBFC-rPPG_FactorizePhys/S002

=== Processing S003 ===
  Video: 2699 frames @ 30 fps
  PPG (wiener): min=-0.0975, max=112.6142
  16 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/ppg/5_demo/UBFC-rPPG_FactorizePhys/S003

=== Processing S004 ===

In [11]:
# PyTorch Dataset + DataLoader
# DATA_FORMAT: NCDHW -> transpose from (T, H, W, C) to (C, T, H, W)

class PPGDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))  # (T,)

        # NDHWC -> NCDHW: transpose (T, H, W, C) -> (C, T, H, W)
        data = np.transpose(data, (3, 0, 1, 2))  # (3, T, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = PPGDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)
print(f"DataLoader ready: {len(loader)} batches")

Dataset: 83 clips
DataLoader ready: 21 batches


In [12]:
# Post-processing functions
# LABEL_TYPE = Standardized -> diff_flag=False (NO cumsum)

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp(pred_chunks, label_chunks, fs=30, diff_flag=False):
    """Post-process BVP predictions and labels.

    Both predicted and label BVP go through the same pipeline:
    detrend -> bandpass -> FFT HR extraction.
    This is the standard rPPG evaluation protocol.

    diff_flag=False for Standardized labels (no cumsum needed).
    """
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    if diff_flag:
        pred  = detrend(np.cumsum(pred),  100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred  = detrend(pred,  100)
        label = detrend(label, 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

In [13]:
# Blink rate detection

def detect_blink_rate(video_path, fps=30, large_box_coef=1.5):
    """Estimate blink rate (blinks/min) using eye-strip brightness analysis.

    Steps:
      1. Detect face on frame 0 with Haar Cascade.
      2. Define eye strip = rows [20%, 50%] of face bbox height.
      3. Compute mean grayscale brightness in eye strip per frame.
      4. Invert (blinks = dips) and bandpass filter [0.1, 0.9] Hz.
      5. Count peaks with minimum separation of 1 second.
    """
    frames = read_video_frames(video_path)
    T = len(frames)

    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(xml_path)
    gray0 = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray0, scaleFactor=1.3, minNeighbors=5)
    H, W = frames[0].shape[:2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H

    eye_top    = y + int(fh * 0.20)
    eye_bottom = y + int(fh * 0.50)
    eye_left   = x
    eye_right  = x + fw

    brightness = np.array([
        np.mean(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY)[eye_top:eye_bottom, eye_left:eye_right])
        for f in frames
    ])

    # blinks = dips in brightness -> invert for find_peaks
    brightness_inv = -brightness.astype(np.float64)

    # bandpass [0.1, 0.9] Hz = [6, 54] blinks/min
    b, a = signal.butter(2, [0.1 / fps * 2, 0.9 / fps * 2], btype="bandpass")
    filtered = signal.filtfilt(b, a, brightness_inv)

    # minimum 1 second between detected blinks
    min_dist = int(fps)
    peaks, _ = signal.find_peaks(filtered, distance=min_dist)

    duration_min = T / fps / 60.0
    return len(peaks) / duration_min


# Compute blink rate for each subject
blink_rate_dict = {}
for subj in subjects:
    subj_key   = subj["subj_key"]
    video_path = subj["video_path"]
    print(f"  {subj_key} ...", end=" ", flush=True)
    blink_rate = detect_blink_rate(video_path, fps=VIDEO_FPS)
    blink_rate_dict[subj_key] = blink_rate
    print(f"{blink_rate:.2f} blinks/min")

print("\nBlink rates:", {k: f"{v:.2f}" for k, v in blink_rate_dict.items()})

  S000 ... 24.63 blinks/min
  S001 ... 31.44 blinks/min
  S002 ... 32.20 blinks/min
  S003 ... 26.01 blinks/min
  S004 ... 30.77 blinks/min

Blink rates: {'S000': '24.63', 'S001': '31.44', 'S002': '32.20', 'S003': '26.01', 'S004': '30.77'}


In [14]:
# Inference: UBFC-rPPG_FactorizePhys
# Ground truth HR = FFT of label BVP (standard rPPG evaluation protocol)

FS = VIDEO_FPS

print(f"Model: {MODEL_LABEL}")
print(f"Path:  {MODEL_PATH}")
print("=" * 60)

# Load model
MD_CONFIG = {
    "FRAME_NUM": CHUNK_LENGTH,
    "MD_FSAM": True,
    "MD_TYPE": "NMF",
    "MD_TRANSFORM": "T_KAB",
    "MD_R": 1,
    "MD_S": 1,
    "MD_STEPS": 3,
    "MD_INFERENCE": True,
    "MD_RESIDUAL": True,
}
model = FactorizePhys(
    frames=CHUNK_LENGTH,
    md_config=MD_CONFIG,
    in_channels=3,
    dropout=0.1,
    device=torch.device(DEVICE),
)

state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
if any(k.startswith("module.") for k in state_dict.keys()):
    state_dict = {k[len("module."):]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)
model = model.to(DEVICE)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

# Run inference
preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray}
labels_dict = {}

with torch.no_grad():
    for batch in tqdm(loader, desc="Inference"):
        data_t, labels_t, batch_subjects, batch_chunk_ids = batch

        data_in = data_t.float().to(DEVICE)  # (N, C, T, H, W) = NCDHW

        # FactorizePhys uses torch.diff internally and needs T+1 frames
        last_frame = data_in[:, :, -1:, :, :].clone()
        data_padded = torch.cat([data_in, last_frame], dim=2)  # (N, 3, T+1, H, W)

        out = model(data_padded)
        pred_ppg = out[0]  # (N, T)

        pred_np  = pred_ppg.cpu().numpy()
        label_np = labels_t.numpy()

        N = data_t.shape[0]
        for i in range(N):
            subj = batch_subjects[i]
            cid  = int(batch_chunk_ids[i])

            if subj not in preds_dict:
                preds_dict[subj]  = {}
                labels_dict[subj] = {}

            preds_dict[subj][cid]  = pred_np[i]
            labels_dict[subj][cid] = label_np[i]

print(f"\nInference complete. Subjects: {sorted(preds_dict.keys())}")

# Per-subject results using BVP-derived HR as ground truth
per_subject_results = []
hr_preds_all = []
hr_labels_all = []
snr_all      = []

print(f"\n{'Subject':<10} {'HR_pred':>10} {'HR_label':>10} {'HR_err':>8} {'SNR':>7} {'Blinks':>7}")
print("-" * 60)

for subj_key in sorted(preds_dict.keys()):
    hr_pred, hr_label, snr_db, pred_processed = process_bvp(
        preds_dict[subj_key], labels_dict[subj_key], fs=FS, diff_flag=False
    )

    blink_rate = blink_rate_dict.get(subj_key, float("nan"))
    hr_err     = hr_pred - hr_label

    # map subj_key ("S000") back to original ID ("S_000")
    subj_id = subj_key[0] + "_" + subj_key[1:]

    per_subject_results.append({
        "name":                subj_id,
        "predicted_heartrate": hr_pred,
        "real_heartrate":      hr_label,
        "heartrate_error":     hr_err,
        "blink_rate":          blink_rate,
        "snr_db":              snr_db,
    })

    hr_preds_all.append(hr_pred)
    hr_labels_all.append(hr_label)
    snr_all.append(snr_db)

    print(f"{subj_id:<10} {hr_pred:>10.3f} {hr_label:>10.3f} {hr_err:>8.3f} {snr_db:>7.2f} {blink_rate:>7.2f}")

hr_preds_all  = np.array(hr_preds_all)
hr_labels_all = np.array(hr_labels_all)
snr_all       = np.array(snr_all)

# Aggregate metrics
n = len(hr_preds_all)
assert n > 0, "No subjects to evaluate."

err   = hr_preds_all - hr_labels_all
abs_e = np.abs(err)
sq_e  = err ** 2
rel_e = abs_e / (np.abs(hr_labels_all) + 1e-9)

mae       = float(np.mean(abs_e))
mae_se    = float(np.std(abs_e) / np.sqrt(n))
rmse      = float(np.sqrt(np.mean(sq_e)))
rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))
mape      = float(np.mean(rel_e) * 100.0)
mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

if n >= 2:
    pearson_r  = float(np.corrcoef(hr_preds_all, hr_labels_all)[0, 1])
    pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
else:
    pearson_r, pearson_se = float("nan"), float("nan")

mean_snr    = float(np.mean(snr_all))
mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

print(f"\nAggregate metrics for {MODEL_LABEL}:")
print(f"  MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
print(f"  RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
print(f"  MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
print(f"  Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
print(f"  SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

# Export results
model_output_dir = os.path.join(OUTPUT_DIR, MODEL_LABEL)
os.makedirs(model_output_dir, exist_ok=True)

metrics_dict = {
    "model":      MODEL_LABEL,
    "n_subjects": n,
    "evaluation_method": "FFT",
    "ground_truth": "BVP-derived HR (FFT of PPG green label)",
    "label_type": LABEL_TYPE,
    "bvp_bandpass_hz":   [0.6, 3.3],
    "aggregate_metrics": {
        "MAE":     {"value": mae,      "se": mae_se,      "unit": "bpm"},
        "RMSE":    {"value": rmse,     "se": rmse_se,     "unit": "bpm"},
        "MAPE":    {"value": mape,     "se": mape_se,     "unit": "%"},
        "Pearson": {"value": pearson_r, "se": pearson_se, "unit": ""},
        "SNR":     {"value": mean_snr, "se": mean_snr_se, "unit": "dB"},
    },
    "per_subject": [
        {
            "name":               r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "real_heartrate":      r["real_heartrate"],
            "heartrate_error":     r["heartrate_error"],
            "snr_db":              r["snr_db"],
        }
        for r in per_subject_results
    ],
}

json_path = os.path.join(model_output_dir, "metrics.json")
with open(json_path, "w") as fh:
    json.dump(metrics_dict, fh, indent=2)
print(f"\n  Metrics saved to: {json_path}")

csv_rows = []
for r in per_subject_results:
    csv_rows.append({
        "name":               r["name"],
        "predicted_heartrate": r["predicted_heartrate"],
        "real_heartrate":      r["real_heartrate"],
        "heartrate_error":     r["heartrate_error"],
        "blink_rate":          r["blink_rate"],
    })

results_df = pd.DataFrame(csv_rows, columns=[
    "name", "predicted_heartrate", "real_heartrate",
    "heartrate_error", "blink_rate"
])

csv_path = os.path.join(model_output_dir, "ppg_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"  CSV saved to: {csv_path}")
print()
print(results_df.to_string(index=False))

# Clean up
del model
torch.cuda.empty_cache()

print(f"\nDone. Results in: {model_output_dir}")

Model: UBFC-rPPG_FactorizePhys
Path:  /home/naver/disk2/HoangDPB/rPPG-Toolbox/final_model_release/UBFC-rPPG_FactorizePhys_FSAM_Res.pth
Total parameters: 52,168


Inference: 100%|██████████| 21/21 [00:01<00:00, 18.12it/s]



Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred   HR_label   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          88.330     83.496    4.834    6.95   24.63
S_001          87.012     57.129   29.883  -10.79   31.44
S_002          71.191     58.008   13.184   -6.90   32.20
S_003          80.420     80.420    0.000    2.89   26.01
S_004          72.510     72.949   -0.439    2.05   30.77

Aggregate metrics for UBFC-rPPG_FactorizePhys:
  MAE     : 9.6680 +/- 4.9920 bpm
  RMSE    : 14.7672 +/- 12.3966 bpm
  MAPE    : 16.2854 +/- 8.8554 %
  Pearson : 0.2816 +/- 0.5540
  SNR     : -1.1590 +/- 2.9539 dB

  Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/ppg/5_demo/UBFC-rPPG_FactorizePhys_wiener/UBFC-rPPG_FactorizePhys/metrics.json
  CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/ppg/5_demo/UBFC-rPPG_FactorizePhys_wiener/UBFC-rPPG_FactorizePhys/ppg_results.csv

  name  